# PCR & PLS Replication: Gu, Kelly & Xiu (2020)
## "Empirical Asset Pricing via Machine Learning"

This notebook runs the principal component regression (PCR) and partial least squares (PLS) replication pipeline. Its opening setup mirrors the neural-network replication notebook: explicit dependency checks, data-location setup, lightweight schema inspection, and an expanding-window split preview.


---
## Step 0: Install Dependencies & Imports


In [18]:
from __future__ import annotations

import argparse
import gc
import json
import os
import re
import time
import warnings
from pathlib import Path

try:
    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq
except ImportError as exc:
    raise ImportError(
        "Missing core packages. Select the project kernel and install them with:\n"
        "%pip install numpy pandas pyarrow"
    ) from exc

warnings.filterwarnings("ignore")

# =====================================================================
# Configuration
# =====================================================================
IN_COLAB = Path("/content").exists()
DRIVE_DATA_DIR = Path(os.environ.get(
    "GKX_DRIVE_DATA_DIR",
    "/content/drive/MyDrive/Replication Paper/Data",
)).expanduser()
PANEL_FILE_NAME = os.environ.get("GKX_PANEL_FILE_NAME", "gkx2020_panel_trimmed.parquet")
PANEL_FILE_CANDIDATES = [
    PANEL_FILE_NAME,
    "gkx2020_panel_trimmed.parquet",
    "gkx2020_panel_trimmed_paper_sample.parquet",
    "gkx2020_panel.parquet",
]

LOCAL_COLAB_DATA_DIR = Path(os.environ.get("GKX_COLAB_DATA_DIR", "/content/data")).expanduser()
DATA_DIR = LOCAL_COLAB_DATA_DIR if IN_COLAB else Path(os.environ.get("GKX_DATA_DIR", "data")).expanduser()
OUTPUT_DIR = Path(os.environ.get("GKX_OUTPUT_DIR", "output/pcr_pls_results_v3")).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PANEL_FILE = DATA_DIR / PANEL_FILE_NAME
FF_FILE = DATA_DIR / "ff5_mom.csv"

FIRST_TEST_YEAR = 1987
LAST_TEST_YEAR = 2021
VALIDATION_YEARS = 12
MAX_K_PCR = 60
MAX_K_PLS = 20
MIN_K = 1
TOP_N_EVAL = 1000
N_DECILES = 10
NW_LAG = 6                          # Newey-West lag for DM and FF regressions
PCR_VI_RECOMPUTE_PCS = False        # False = GKX spec (freeze loadings); True = slower variant
K_ENSEMBLE_DELTA = 0                # 0 disables; try 3 for K-ensemble smoothing
MARGINAL_CHARS = ["mvel1", "mom12m", "retvol", "acc"]
META_COLS = {"permno", "date", "ret_excess", "me", "rf", "exchcd", "split", "ret"}
EPS = 1e-12

# ====================================================
# SMOKE TEST MODE
# Set to True for a quick sanity check. It reduces the
# number of refit years and component ceilings.
# Set to False for the full replication.
# ====================================================
SMOKE_TEST = False

if SMOKE_TEST:
    LAST_TEST_YEAR = min(LAST_TEST_YEAR, FIRST_TEST_YEAR + 1)
    MAX_K_PCR = min(MAX_K_PCR, 10)
    MAX_K_PLS = min(MAX_K_PLS, 5)
    TOP_N_EVAL = min(TOP_N_EVAL, 250)
    print("\n** SMOKE TEST MODE - reduced settings for quick validation **")

print(f"IN_COLAB:   {IN_COLAB}")
print(f"DATA_DIR:   {DATA_DIR}")
print(f"PANEL_NAME: {PANEL_FILE_NAME}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"PANEL_FILE configured: {PANEL_FILE}")
print(f"FF_FILE configured:    {FF_FILE}")
print(f"SMOKE_TEST: {SMOKE_TEST}")


IN_COLAB:   True
DATA_DIR:   /content/data
PANEL_NAME: gkx2020_panel_trimmed.parquet
OUTPUT_DIR: output/pcr_pls_results_v3
PANEL_FILE configured: /content/data/gkx2020_panel_trimmed.parquet
FF_FILE configured:    /content/data/ff5_mom.csv
SMOKE_TEST: False


---
## Step 1: Locate or Stage Data


In [19]:
# ====================================================
# Google Drive -> Colab runtime staging
# In Colab, this notebook mounts Drive, copies the trimmed panel into
# /content/data, and then reads the local runtime copy for speed.
# ====================================================


def mount_google_drive_if_colab():
    if not IN_COLAB:
        return
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("This notebook is running in /content but google.colab is unavailable.") from exc

    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.exists():
        drive.mount("/content/drive")


def resolve_panel_file(data_dir):
    candidates = []
    for name in PANEL_FILE_CANDIDATES:
        candidate = data_dir / name
        if candidate not in candidates:
            candidates.append(candidate)

    for candidate in candidates:
        if candidate.exists():
            return candidate

    searched = "\n  - ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Could not find a GKX panel parquet. Searched:\n"
        f"  - {searched}\n"
        "Set GKX_PANEL_FILE_NAME or GKX_DRIVE_DATA_DIR if your file uses another name/path."
    )


def stage_panel_to_colab_runtime():
    if not IN_COLAB:
        return resolve_panel_file(DATA_DIR)

    import shutil

    mount_google_drive_if_colab()
    drive_panel = resolve_panel_file(DRIVE_DATA_DIR)
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    local_panel = DATA_DIR / drive_panel.name
    needs_copy = (not local_panel.exists()) or (local_panel.stat().st_size != drive_panel.stat().st_size)
    if needs_copy:
        print(f"Copying panel from Drive to Colab runtime:\n  {drive_panel}\n  -> {local_panel}")
        shutil.copy2(drive_panel, local_panel)
    else:
        print(f"Colab runtime copy already exists: {local_panel}")

    drive_ff = DRIVE_DATA_DIR / "ff5_mom.csv"
    local_ff = DATA_DIR / "ff5_mom.csv"
    if drive_ff.exists():
        ff_needs_copy = (not local_ff.exists()) or (local_ff.stat().st_size != drive_ff.stat().st_size)
        if ff_needs_copy:
            print(f"Copying FF factor file to Colab runtime: {local_ff}")
            shutil.copy2(drive_ff, local_ff)
    else:
        print(f"Optional FF factor file not found in Drive: {drive_ff}")

    return local_panel


def describe_panel_file(panel_file):
    pq_file = pq.ParquetFile(panel_file)
    columns = list(pq_file.schema.names)
    meta_cols = [c for c in ["permno", "date", "ret_excess", "me", "exchcd"] if c in columns]
    meta_df = pd.read_parquet(panel_file, columns=meta_cols)
    dates = pd.to_datetime(meta_df["date"])
    file_size_gb = panel_file.stat().st_size / 1024**3
    feature_count = len([c for c in columns if c not in META_COLS])

    print("\nPanel quick info")
    print("-" * 70)
    print(f"File: {panel_file}")
    print(f"Rows: {len(meta_df):,}")
    print(f"Columns: {len(columns):,} ({feature_count:,} features + {len(columns) - feature_count:,} metadata)")
    print(f"Date range: {dates.min().date()} to {dates.max().date()}")
    print(f"Unique stocks: {meta_df['permno'].nunique():,}" if "permno" in meta_df else "Unique stocks: n/a")
    print(f"Parquet row groups: {pq_file.num_row_groups:,}")
    print(f"File size: {file_size_gb:.2f} GB")
    if "exchcd" in meta_df:
        exch_counts = {
            (int(k) if pd.notna(k) else "NaN"): int(v)
            for k, v in meta_df["exchcd"].value_counts(dropna=False).sort_index().items()
        }
        print(f"Exchange counts: {exch_counts}")
    print("-" * 70)


mount_google_drive_if_colab()
PANEL_FILE = stage_panel_to_colab_runtime()
DATA_DIR = PANEL_FILE.parent
FF_FILE = DATA_DIR / "ff5_mom.csv"
if IN_COLAB and "GKX_OUTPUT_DIR" not in os.environ:
    OUTPUT_DIR = DRIVE_DATA_DIR / "Results" / "pcr_pls"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Panel file ready: {PANEL_FILE}  (exists={PANEL_FILE.exists()})")
print(f"FF factor file:  {FF_FILE}  (exists={FF_FILE.exists()})")
print(f"Output dir ready: {OUTPUT_DIR}")
print("Reading panel from the Colab runtime copy." if IN_COLAB else "Reading panel from local DATA_DIR.")
describe_panel_file(PANEL_FILE)


Colab runtime copy already exists: /content/data/gkx2020_panel_trimmed.parquet
Optional FF factor file not found in Drive: /content/drive/MyDrive/Replication Paper/Data/ff5_mom.csv
Panel file ready: /content/data/gkx2020_panel_trimmed.parquet  (exists=True)
FF factor file:  /content/data/ff5_mom.csv  (exists=False)
Output dir ready: /content/drive/MyDrive/Replication Paper/Data/Results/pcr_pls
Reading panel from the Colab runtime copy.

Panel quick info
----------------------------------------------------------------------
File: /content/data/gkx2020_panel_trimmed.parquet
Rows: 3,305,648
Columns: 283 (278 features + 5 metadata)
Date range: 1957-03-01 to 2021-12-01
Unique stocks: 25,770
Parquet row groups: 4
File size: 1.96 GB
Exchange counts: {1: 1063121, 2: 432944, 3: 1809583}
----------------------------------------------------------------------


---
## Step 2: Identify Feature Columns & Target


In [20]:
# ====================================================
# Lightweight schema inspection. This mirrors the NN
# notebook feature breakdown without loading the full panel.
# ====================================================
TARGET_COL = "ret_excess"


def inspect_panel_schema(panel_file=PANEL_FILE):
    if not panel_file.exists():
        print(f"Panel file not found yet: {panel_file}")
        return []

    pq_file = pq.ParquetFile(panel_file)
    all_columns = list(pq_file.schema.names)
    feature_cols = [c for c in all_columns if c not in META_COLS]

    char_cols = [c for c in feature_cols if not c.startswith("ind_") and "_x_" not in c]
    interaction_cols = [c for c in feature_cols if "_x_" in c]
    industry_cols = [c for c in feature_cols if c.startswith("ind_")]

    print(f"Total columns: {len(all_columns)}")
    print(f"Total features: {len(feature_cols)}")
    print(f"  - Firm characteristics: {len(char_cols)}")
    print(f"  - Char x macro interactions: {len(interaction_cols)}")
    print(f"  - Industry dummies: {len(industry_cols)}")
    print(f"Target: {TARGET_COL}")

    return feature_cols


PANEL_FEATURE_COLS = inspect_panel_schema()


Total columns: 283
Total features: 278
  - Firm characteristics: 94
  - Char x macro interactions: 94
  - Industry dummies: 90
Target: ret_excess


---
## Step 3: Define Sample Splits (Expanding Window)


In [21]:
# ====================================================
# Expanding-window split preview. The PCR/PLS implementation
# below uses the same rules directly from year_to_slice after
# loading the panel.
# ====================================================
def build_split_preview(first_test_year=FIRST_TEST_YEAR, last_test_year=LAST_TEST_YEAR, validation_years=VALIDATION_YEARS):
    splits = []
    for test_year in range(first_test_year, last_test_year + 1):
        train_end_year = test_year - validation_years - 1
        valid_start_year = train_end_year + 1
        valid_end_year = test_year - 1
        splits.append({
            "train": ("panel_start", train_end_year),
            "valid": (valid_start_year, valid_end_year),
            "test": (test_year, test_year),
        })
    return splits


SPLIT_PREVIEW = build_split_preview()

print(f"Number of annual refits: {len(SPLIT_PREVIEW)}")
if SPLIT_PREVIEW:
    print(f"First split: train {SPLIT_PREVIEW[0]['train']}, valid {SPLIT_PREVIEW[0]['valid']}, test {SPLIT_PREVIEW[0]['test']}")
    print(f"Last split:  train {SPLIT_PREVIEW[-1]['train']}, valid {SPLIT_PREVIEW[-1]['valid']}, test {SPLIT_PREVIEW[-1]['test']}")


Number of annual refits: 35
First split: train ('panel_start', 1974), valid (1975, 1986), test (1987, 1987)
Last split:  train ('panel_start', 2008), valid (2009, 2020), test (2021, 2021)


---
## Step 4: PCR/PLS Implementation


In [22]:
# =====================================================================
# 1. Panel loading (memory-efficient, streaming column batches)
# =====================================================================
def load_panel():
    """Load the panel in (year, month, permno) sorted order with minimum
    peak memory. Returns a dict of numpy arrays; the large X_all is float32."""
    print("\n" + "=" * 70)
    print("LOADING PANEL")
    print("=" * 70)

    pq_file = pq.ParquetFile(PANEL_FILE)
    meta_cols = ["permno", "date", "ret_excess", "me"]
    has_exchcd = "exchcd" in pq_file.schema.names
    if has_exchcd:
        meta_cols.append("exchcd")

    meta_df = pd.read_parquet(PANEL_FILE, columns=meta_cols)
    meta_df["date"] = pd.to_datetime(meta_df["date"]).dt.to_period("M")
    N = len(meta_df)
    print(f"  rows       : {N:,}")

    yr = meta_df["date"].dt.year.to_numpy(dtype=np.int32)
    mo = meta_df["date"].dt.month.to_numpy(dtype=np.int32)
    pm = meta_df["permno"].to_numpy()
    sort_idx = np.lexsort((pm, mo, yr))
    already_sorted = np.array_equal(sort_idx, np.arange(N))
    print(f"  pre-sorted : {already_sorted}")

    if already_sorted:
        y_all = meta_df["ret_excess"].to_numpy(dtype=np.float64)
        me_all = meta_df["me"].to_numpy(dtype=np.float64)
        permno_all = pm
        exchcd_all = meta_df["exchcd"].to_numpy() if has_exchcd else np.full(N, np.nan)
    else:
        y_all = meta_df["ret_excess"].to_numpy(dtype=np.float64)[sort_idx]
        me_all = meta_df["me"].to_numpy(dtype=np.float64)[sort_idx]
        permno_all = pm[sort_idx]
        exchcd_all = meta_df["exchcd"].to_numpy()[sort_idx] if has_exchcd else np.full(N, np.nan)
        yr = yr[sort_idx]
        mo = mo[sort_idx]

    del meta_df
    gc.collect()

    feature_cols = [c for c in pq_file.schema.names if c not in META_COLS]
    P = len(feature_cols)
    print(f"  features   : P={P}")
    print(f"  X_all size : {N*P*4/1024**3:.2f} GB (float32)")

    X_all = np.empty((N, P), dtype=np.float32)
    BATCH = 50
    for bs in range(0, P, BATCH):
        be = min(bs + BATCH, P)
        batch = pd.read_parquet(PANEL_FILE, columns=feature_cols[bs:be])
        for k, col in enumerate(feature_cols[bs:be]):
            src = batch[col].to_numpy(dtype=np.float32, copy=False)
            X_all[:, bs + k] = src if already_sorted else src[sort_idx]
        del batch
        gc.collect()
        print(f"    {be}/{P} columns loaded", end="\r")
    print()

    date_str = np.array([f"{y:04d}-{m:02d}" for y, m in zip(yr, mo)])
    feat_arr = np.array(feature_cols)

    year_to_slice = {}
    for y in np.unique(yr):
        rows = np.where(yr == y)[0]
        year_to_slice[int(y)] = (int(rows[0]), int(rows[-1]) + 1)

    print(f"  years      : {min(year_to_slice)}-{max(year_to_slice)}")
    print(f"  stocks     : {np.unique(permno_all).size:,}")

    return dict(
        X_all=X_all, y_all=y_all, me_all=me_all, exchcd_all=exchcd_all,
        permno_all=permno_all, panel_year=yr, panel_month=mo,
        date_str_all=date_str, feature_cols_arr=feat_arr,
        year_to_slice=year_to_slice, N=N, P=P,
    )


def slice_up_to_year(yts, year):
    valid = [y for y in yts if y <= year]
    if not valid:
        return None
    return (yts[min(yts)][0], yts[max(valid)][1])


def slice_year_range(yts, a, b):
    valid = [y for y in yts if a <= y <= b]
    if not valid:
        return None
    return (yts[min(valid)][0], yts[max(valid)][1])


# =====================================================================
# 2. Core primitives
# =====================================================================
def r2_oos(y_true, y_pred):
    """GKX convention: 1 - sum((y - yhat)^2) / sum(y^2). Zero benchmark."""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum(y_true ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > EPS else np.nan


def fit_standardizer(X_train):
    mu = X_train.mean(axis=0, dtype=np.float64).astype(np.float32)
    sigma = X_train.std(axis=0, dtype=np.float64).astype(np.float32)
    sigma = np.where(sigma < EPS, 1.0, sigma)
    return mu, sigma


def standardize_inplace(X, mu, sigma):
    """Allocate once, subtract and divide in-place. Returns a new float32 array."""
    out = X - mu      # broadcasts; creates new array
    out /= sigma
    return out


def fit_pcr(X_std, max_k):
    """PCR loadings = top-K eigenvectors of X'X. Gram-matrix route.
    For N >> P this is O(N*P^2), vastly cheaper than SVD of X which is O(N^2*P)."""
    n, p = X_std.shape
    k_eff = min(max_k, p, n - 1)
    if k_eff <= 0:
        return np.zeros((p, 0), dtype=np.float32), 0
    XtX = (X_std.T @ X_std).astype(np.float64)
    eigvals, eigvecs = np.linalg.eigh(XtX)
    order = np.argsort(eigvals)[::-1][:k_eff]
    return eigvecs[:, order].astype(np.float32), k_eff


def fit_pls(X_std, y, max_k):
    """SIMPLS (de Jong 1993) operating in P-space.
    Verified against sklearn.cross_decomposition.PLSRegression at corr=1.0."""
    n, p = X_std.shape
    k_eff = min(max_k, p, n - 1)
    if k_eff <= 0:
        return np.zeros((p, 0), dtype=np.float32), 0

    y_c = (y - y.mean()).astype(np.float64)
    s = X_std.T @ y_c
    S = (X_std.T @ X_std).astype(np.float64)

    R = np.zeros((p, k_eff), dtype=np.float64)
    V = np.zeros((p, k_eff), dtype=np.float64)
    kept = 0

    for h in range(k_eff):
        w = s.copy()
        n_w = np.linalg.norm(w)
        if n_w < EPS:
            break
        w /= n_w
        Sw = S @ w
        d = float(w @ Sw)
        if d < EPS:
            break
        p_load = Sw / d
        if h > 0:
            p_load = p_load - V[:, :h] @ (V[:, :h].T @ p_load)
        n_p = np.linalg.norm(p_load)
        if n_p < EPS:
            break
        v = p_load / n_p
        V[:, h] = v
        R[:, h] = w
        s = s - v * float(v @ s)
        kept += 1

    return R[:, :kept].astype(np.float32), kept


def tune_k(loadings, X_tr_std, y_tr, X_va_std, y_va, min_k=MIN_K):
    """Sweep K from 1..K_max on the validation set; return arg-max K, the
    val R² at K*, and the full R²(K) curve for diagnostic output."""
    k_max = loadings.shape[1]
    if k_max == 0:
        return 0, 0.0, np.array([0.0]), np.array([1.0])  # curve at K=0 is "R²=0 exactly"

    S_tr = (X_tr_std @ loadings).astype(np.float64)
    S_va = (X_va_std @ loadings).astype(np.float64)
    StS = S_tr.T @ S_tr
    Sty = S_tr.T @ y_tr.astype(np.float64)
    ss_tot_va = float(np.sum(y_va ** 2))

    r2_curve = np.full(k_max + 1, np.nan)
    r2_curve[0] = 0.0  # K=0 ⇒ predict zero ⇒ R²_oos = 0 by construction

    best_k, best_r2 = 0, 0.0
    for k in range(1, k_max + 1):
        try:
            beta = np.linalg.solve(StS[:k, :k], Sty[:k])
        except np.linalg.LinAlgError:
            beta, *_ = np.linalg.lstsq(S_tr[:, :k], y_tr, rcond=None)
        y_pred = S_va[:, :k] @ beta
        ss_res = float(np.sum((y_va - y_pred) ** 2))
        r2 = 1.0 - ss_res / ss_tot_va if ss_tot_va > EPS else -np.inf
        r2_curve[k] = r2
        if k >= min_k and r2 > best_r2 + 1e-15:
            best_r2, best_k = r2, k

    return best_k, best_r2, r2_curve, None   # (trailing slot reserved for future use)


# =====================================================================
# 3. Recursive backtest with optional K-ensemble
# =====================================================================
def predict_single_k(X_std_te, loadings, y_tr, X_tr_std, k):
    """OLS on the first k component scores. Returns (y_pred, beta) always."""
    if k == 0:
        return (np.zeros(X_std_te.shape[0], dtype=np.float64),
                np.zeros(0, dtype=np.float64))
    S_tr = (X_tr_std @ loadings[:, :k]).astype(np.float64)
    beta, *_ = np.linalg.lstsq(S_tr, y_tr, rcond=None)
    S_te = (X_std_te @ loadings[:, :k]).astype(np.float64)
    return S_te @ beta, beta


def predict_kensemble(X_std_te, loadings, y_tr, X_tr_std,
                      best_k, r2_curve, delta=K_ENSEMBLE_DELTA):
    """Forecast as a val-R²-weighted average over K ∈ [best_k - delta, best_k + delta].
    Negative-R² K's get zero weight. If delta=0 this reduces to single-K prediction."""
    if delta == 0 or best_k == 0:
        y_pred, beta = predict_single_k(X_std_te, loadings, y_tr, X_tr_std, best_k)
        return y_pred, beta, best_k

    k_lo = max(MIN_K, best_k - delta)
    k_hi = min(loadings.shape[1], best_k + delta)
    weights, preds = [], []
    for k in range(k_lo, k_hi + 1):
        r2k = r2_curve[k] if k < len(r2_curve) else -np.inf
        if np.isfinite(r2k) and r2k > 0:
            yp, _ = predict_single_k(X_std_te, loadings, y_tr, X_tr_std, k)
            weights.append(r2k)
            preds.append(yp)
    if not preds:
        y_pred, beta = predict_single_k(X_std_te, loadings, y_tr, X_tr_std, best_k)
        return y_pred, beta, best_k
    weights = np.array(weights) / np.sum(weights)
    y_pred = np.sum([w * p for w, p in zip(weights, preds)], axis=0)
    # beta returned for the central (best_k) model — used for VI downstream
    _, beta = predict_single_k(X_std_te, loadings, y_tr, X_tr_std, best_k)
    return y_pred, beta, best_k


def run_one_window(model_name, panel, tr_slc, va_slc, te_slc, max_k,
                   k_ensemble_delta=K_ENSEMBLE_DELTA):
    X_all, y_all = panel["X_all"], panel["y_all"]
    X_tr = X_all[tr_slc[0]:tr_slc[1]]
    y_tr = y_all[tr_slc[0]:tr_slc[1]]
    X_va = X_all[va_slc[0]:va_slc[1]]
    y_va = y_all[va_slc[0]:va_slc[1]]
    X_te = X_all[te_slc[0]:te_slc[1]]

    mu, sigma = fit_standardizer(X_tr)
    X_tr_std = standardize_inplace(X_tr, mu, sigma)
    X_va_std = standardize_inplace(X_va, mu, sigma)
    X_te_std = standardize_inplace(X_te, mu, sigma)

    if model_name == "pcr":
        loadings, _ = fit_pcr(X_tr_std, max_k)
    else:
        loadings, _ = fit_pls(X_tr_std, y_tr, max_k)

    best_k, val_r2, r2_curve, _ = tune_k(loadings, X_tr_std, y_tr, X_va_std, y_va)

    y_pred_te, beta, k_used = predict_kensemble(
        X_te_std, loadings, y_tr, X_tr_std, best_k, r2_curve, delta=k_ensemble_delta)

    result = dict(
        best_k=best_k, k_used=k_used, val_r2=val_r2,
        r2_curve=r2_curve,
        y_pred_te=y_pred_te,
        loadings=loadings, beta=beta, mu=mu, sigma=sigma,
        tr_slc=tr_slc, va_slc=va_slc,
        ceiling_hit=(best_k == loadings.shape[1] and loadings.shape[1] > 0),
    )

    # Explicit cleanup: these each hold several GB at P=920
    del X_tr_std, X_va_std, X_te_std
    gc.collect()
    return result


def run_backtest(model_name, max_k, panel, k_ensemble_delta=K_ENSEMBLE_DELTA):
    print(f"\n{'='*70}\nRunning {model_name.upper()} (max K = {max_k}, "
          f"K-ensemble delta = {k_ensemble_delta})\n{'='*70}")

    yts = panel["year_to_slice"]
    preds_list, yearly_log, r2_curves = [], [], []
    last_window, nonzero_windows = None, []

    for test_year in range(FIRST_TEST_YEAR, LAST_TEST_YEAR + 1):
        t0 = time.time()
        tr = slice_up_to_year(yts, test_year - VALIDATION_YEARS - 1)
        va = slice_year_range(yts, test_year - VALIDATION_YEARS, test_year - 1)
        te = slice_year_range(yts, test_year, test_year)
        if tr is None or va is None or te is None:
            continue

        out = run_one_window(model_name, panel, tr, va, te, max_k, k_ensemble_delta)

        te_idx = np.arange(te[0], te[1])
        preds_list.append(dict(
            permno=panel["permno_all"][te_idx],
            date=panel["date_str_all"][te_idx],
            year=panel["panel_year"][te_idx],
            month=panel["panel_month"][te_idx],
            exchcd=panel["exchcd_all"][te_idx],
            ret_excess=panel["y_all"][te_idx],
            pred=out["y_pred_te"],
            me=panel["me_all"][te_idx],
        ))
        yearly_log.append(dict(
            test_year=test_year,
            best_k=out["best_k"],
            val_r2_pct=out["val_r2"] * 100,
            ceiling_hit=out["ceiling_hit"],
            n_train=tr[1] - tr[0], n_valid=va[1] - va[0], n_test=te[1] - te[0],
        ))
        # Record full val-R² curve for diagnostics
        curve = out["r2_curve"]
        for k, r2 in enumerate(curve):
            if np.isfinite(r2):
                r2_curves.append(dict(test_year=test_year, K=int(k), val_r2_pct=r2*100))

        if out["best_k"] > 0:
            nonzero_windows.append({**out, "test_year": test_year})
        if test_year == LAST_TEST_YEAR:
            last_window = out
            last_window["test_year"] = test_year

        print(f"  [{model_name.upper()}] {test_year} | K*={out['best_k']:3d} | "
              f"val_R²={out['val_r2']*100:+.4f}% | ceiling={out['ceiling_hit']!s:5s} | "
              f"{time.time()-t0:5.1f}s")

    preds_df = pd.DataFrame({k: np.concatenate([p[k] for p in preds_list])
                             for k in preds_list[0].keys()})
    yearly_df = pd.DataFrame(yearly_log)
    r2_curves_df = pd.DataFrame(r2_curves)
    return preds_df, yearly_df, r2_curves_df, last_window, nonzero_windows


# =====================================================================
# 4. Evaluation — Table 1 (panel + top/bot 1000) and Table 2 calendar
# =====================================================================
def evaluate_table1(preds_df, model_name, top_n=TOP_N_EVAL):
    full = r2_oos(preds_df["ret_excess"].values, preds_df["pred"].values)
    top_mask = preds_df.groupby("date")["me"].rank(method="first", ascending=False) <= top_n
    bot_mask = preds_df.groupby("date")["me"].rank(method="first", ascending=True) <= top_n
    top, bot = preds_df[top_mask], preds_df[bot_mask]
    return pd.DataFrame([
        dict(model=model_name, subsample="full_panel",   r2_oos_pct=full*100, n_obs=len(preds_df)),
        dict(model=model_name, subsample=f"top_{top_n}", r2_oos_pct=r2_oos(top["ret_excess"].values, top["pred"].values)*100, n_obs=len(top)),
        dict(model=model_name, subsample=f"bot_{top_n}", r2_oos_pct=r2_oos(bot["ret_excess"].values, bot["pred"].values)*100, n_obs=len(bot)),
    ])


def evaluate_annual_calendar(preds_df, model_name):
    rows = []
    for y, g in preds_df.groupby("year"):
        rows.append(dict(year=int(y), model=model_name,
                         r2_oos_pct=r2_oos(g["ret_excess"].values, g["pred"].values)*100,
                         n_obs=len(g)))
    return pd.DataFrame(rows)


# =====================================================================
# 5. 12-month-ahead pipeline (GKX Table 2) — separate recursive backtest
# =====================================================================
def build_annual_y(panel):
    """Forward-12-month sum of excess returns per stock. NaN where the window
    crosses a stock's exit."""
    order = np.lexsort((panel["panel_month"], panel["panel_year"], panel["permno_all"]))
    inv = np.argsort(order)
    permno_s = panel["permno_all"][order]
    ret_s = panel["y_all"][order]

    n = len(ret_s)
    out = np.full(n, np.nan, dtype=np.float64)
    changes = np.r_[0, np.where(np.diff(permno_s) != 0)[0] + 1, n]
    for i in range(len(changes) - 1):
        s, e = changes[i], changes[i + 1]
        seg = ret_s[s:e]
        k = len(seg)
        if k < 13:
            continue
        cs = np.concatenate(([0.0], np.cumsum(seg)))
        valid = k - 12
        out[s:s + valid] = cs[13:k + 1] - cs[1:valid + 1]
    return out[inv]


def run_backtest_annual(model_name, max_k, panel, y_ann):
    print(f"\n{'='*70}\nRunning {model_name.upper()} ANNUAL HORIZON (max K={max_k})\n{'='*70}")
    yts = panel["year_to_slice"]
    X_all = panel["X_all"]
    preds_list, yearly_log = [], []

    for test_year in range(FIRST_TEST_YEAR, LAST_TEST_YEAR + 1):
        t0 = time.time()
        tr = slice_up_to_year(yts, test_year - VALIDATION_YEARS - 1)
        va = slice_year_range(yts, test_year - VALIDATION_YEARS, test_year - 1)
        te = slice_year_range(yts, test_year, test_year)
        if tr is None or va is None or te is None:
            continue

        idx_tr = np.arange(tr[0], tr[1]); idx_tr = idx_tr[~np.isnan(y_ann[idx_tr])]
        idx_va = np.arange(va[0], va[1]); idx_va = idx_va[~np.isnan(y_ann[idx_va])]
        idx_te = np.arange(te[0], te[1])

        if len(idx_tr) == 0 or len(idx_va) == 0:
            continue

        X_tr = X_all[idx_tr]; y_tr = y_ann[idx_tr]
        X_va = X_all[idx_va]; y_va = y_ann[idx_va]
        X_te = X_all[idx_te]; y_te = y_ann[idx_te]

        mu, sigma = fit_standardizer(X_tr)
        X_tr_std = standardize_inplace(X_tr, mu, sigma)
        X_va_std = standardize_inplace(X_va, mu, sigma)
        X_te_std = standardize_inplace(X_te, mu, sigma)

        if model_name == "pcr":
            loadings, _ = fit_pcr(X_tr_std, max_k)
        else:
            loadings, _ = fit_pls(X_tr_std, y_tr, max_k)
        best_k, val_r2, r2_curve, _ = tune_k(loadings, X_tr_std, y_tr, X_va_std, y_va)

        y_pred, _, _ = predict_kensemble(X_te_std, loadings, y_tr, X_tr_std,
                                          best_k, r2_curve, delta=K_ENSEMBLE_DELTA)

        mask = np.isfinite(y_te)
        preds_list.append(dict(
            year=np.full(int(mask.sum()), test_year, dtype=np.int32),
            ret_excess=y_te[mask], pred=y_pred[mask],
        ))
        yearly_log.append(dict(test_year=test_year, best_k=best_k,
                               val_r2_pct=val_r2*100,
                               n_train=len(idx_tr), n_valid=len(idx_va),
                               n_test=int(mask.sum())))
        print(f"  [{model_name.upper()}-ANN] {test_year} | K*={best_k:3d} | "
              f"val_R²={val_r2*100:+.4f}% | n_te={int(mask.sum()):>6,} | {time.time()-t0:5.1f}s")

        del X_tr_std, X_va_std, X_te_std
        gc.collect()

    preds_df = pd.DataFrame({k: np.concatenate([p[k] for p in preds_list])
                             for k in preds_list[0].keys()})
    yearly_df = pd.DataFrame(yearly_log)

    rows = []
    for y, g in preds_df.groupby("year"):
        rows.append(dict(year=int(y), model=model_name,
                         r2_oos_pct=r2_oos(g["ret_excess"].values, g["pred"].values)*100,
                         n_obs=len(g)))
    overall = r2_oos(preds_df["ret_excess"].values, preds_df["pred"].values) * 100
    rows.append(dict(year=-1, model=model_name, r2_oos_pct=overall, n_obs=len(preds_df)))
    return pd.DataFrame(rows), yearly_df


# =====================================================================
# 6. Decile portfolios
# =====================================================================
def decile_portfolio(preds_df, model_name):
    df = preds_df.copy()
    df["_sd"] = df.groupby("date")["pred"].transform("std")
    df = df[df["_sd"] > 0].drop(columns="_sd")
    df["decile"] = df.groupby("date")["pred"].transform(
        lambda x: pd.qcut(x, N_DECILES, labels=False, duplicates="drop"))
    df = df.dropna(subset=["decile"])
    df["decile"] = df["decile"].astype(int) + 1

    def vw(g):
        w = g["me"].values
        ws = w.sum()
        return float(np.sum(g["ret_excess"].values * w) / ws) if ws > 0 else np.nan

    monthly = df.groupby(["date", "decile"]).apply(vw).unstack(level="decile")
    monthly.columns = [f"D{int(c)}" for c in monthly.columns]
    if "D1" in monthly and f"D{N_DECILES}" in monthly:
        monthly["HML"] = monthly[f"D{N_DECILES}"] - monthly["D1"]

    stats = pd.DataFrame({
        "Avg_monthly_pct": monthly.mean() * 100,
        "SD_monthly_pct":  monthly.std()  * 100,
        "SR_annualized":   (monthly.mean() / monthly.std()) * np.sqrt(12),
        "n_months":        monthly.count(),
    }).reset_index().rename(columns={"index": "decile"})
    stats.insert(0, "model", model_name)
    return stats, monthly


# =====================================================================
# 7. Variable importance (PCR + PLS)
# =====================================================================
def variable_importance_pcr(last_window, X_all, y_all, feature_cols_arr,
                             recompute_pcs=PCR_VI_RECOMPUTE_PCS):
    """GKX Section 1.9 definition (default): 'setting all values of predictor j
    to zero, while holding the remaining model estimates fixed.' That means
    LOADINGS STAY FIXED; we only zero column j of X and remeasure R².

    The alternative (recompute_pcs=True) recomputes PCA after zeroing j. This
    is slower and NOT what the paper actually does; kept as a diagnostic."""
    if last_window is None or last_window["best_k"] == 0:
        print("  [PCR-VI] last window has K=0 — VI undefined at final refit.")
        return pd.DataFrame(columns=["feature", "vi_drop_r2", "vi_normalized"])

    P = len(feature_cols_arr)
    tr, va = last_window["tr_slc"], last_window["va_slc"]
    best_k, mu, sigma = last_window["best_k"], last_window["mu"], last_window["sigma"]

    X_tr_std = standardize_inplace(X_all[tr[0]:tr[1]], mu, sigma)
    X_va_std = standardize_inplace(X_all[va[0]:va[1]], mu, sigma)
    y_tr = y_all[tr[0]:tr[1]]
    y_va = y_all[va[0]:va[1]]

    loadings = last_window["loadings"][:, :best_k]
    S_tr = (X_tr_std @ loadings).astype(np.float64)
    S_va = (X_va_std @ loadings).astype(np.float64)
    beta, *_ = np.linalg.lstsq(S_tr, y_tr, rcond=None)
    r2_base = r2_oos(y_va, S_va @ beta)
    print(f"  [PCR-VI] baseline val R² = {r2_base*100:+.4f}%")

    vi = np.zeros(P, dtype=np.float64)
    t0 = time.time()
    for j in range(P):
        saved_tr, saved_va = X_tr_std[:, j].copy(), X_va_std[:, j].copy()
        X_tr_std[:, j] = 0.0
        X_va_std[:, j] = 0.0

        if recompute_pcs:
            # Slower GKX-variant: recompute the basis after zeroing j
            loadings_j, _ = fit_pcr(X_tr_std, best_k)
            S_tr_j = (X_tr_std @ loadings_j).astype(np.float64)
            S_va_j = (X_va_std @ loadings_j).astype(np.float64)
            beta_j, *_ = np.linalg.lstsq(S_tr_j, y_tr, rcond=None)
        else:
            # GKX-faithful: loadings fixed, just re-score and refit OLS
            S_tr_j = (X_tr_std @ loadings).astype(np.float64)
            S_va_j = (X_va_std @ loadings).astype(np.float64)
            beta_j, *_ = np.linalg.lstsq(S_tr_j, y_tr, rcond=None)
        vi[j] = r2_base - r2_oos(y_va, S_va_j @ beta_j)

        X_tr_std[:, j], X_va_std[:, j] = saved_tr, saved_va
        if (j + 1) % 100 == 0:
            print(f"    [PCR-VI] {j+1}/{P}  elapsed={time.time()-t0:.0f}s")

    del X_tr_std, X_va_std
    gc.collect()

    vi_pos = np.clip(vi, 0.0, None)
    total = vi_pos.sum()
    vi_norm = vi_pos / total if total > 0 else vi_pos
    return pd.DataFrame({"feature": feature_cols_arr,
                         "vi_drop_r2": vi,
                         "vi_normalized": vi_norm}
                        ).sort_values("vi_normalized", ascending=False).reset_index(drop=True)


def variable_importance_pls(nonzero_windows, feature_cols_arr):
    """Average |R_j · beta| across EVERY window with best_k > 0, not just the
    most recent one. Uses more information and is robust to the 2016 K=0 case."""
    if not nonzero_windows:
        print("  [PLS-VI] no window has best_k > 0 — empty frame.")
        return pd.DataFrame(columns=["feature", "vi_normalized"])

    contribs = []
    for w in nonzero_windows:
        R = w["loadings"][:, :w["best_k"]]
        beta = w["beta"]
        contribs.append(np.abs(R @ beta))
    contrib = np.mean(contribs, axis=0)
    total = contrib.sum()
    vi_norm = contrib / total if total > 0 else contrib
    return pd.DataFrame({"feature": feature_cols_arr,
                         "vi_normalized": vi_norm}
                        ).sort_values("vi_normalized", ascending=False).reset_index(drop=True)


# =====================================================================
# 8. Aggregate feature-level VI to characteristic / macro level
# =====================================================================
_INTERACTION_RE = re.compile(r"^(.+?)_x_(.+?)_macro$")
_INDUSTRY_PREFIXES = ("sic2_", "ind_", "sic_")

def parse_feature_name(name):
    """Return (base_char_or_flag, macro_or_None, is_industry)."""
    if name.startswith(_INDUSTRY_PREFIXES):
        return ("__industry__", None, True)
    m = _INTERACTION_RE.match(name)
    if m:
        return (m.group(1), m.group(2), False)
    return (name, None, False)


def aggregate_vi_to_char(vi_df, vi_col="vi_normalized"):
    out = []
    for _, r in vi_df.iterrows():
        base, _, is_ind = parse_feature_name(r["feature"])
        out.append({"char": "sic2_industry" if is_ind else base, "vi": r[vi_col]})
    s = pd.DataFrame(out).groupby("char")["vi"].sum()
    tot = s.sum()
    if tot > 0:
        s = s / tot
    return s.sort_values(ascending=False).to_frame("vi_normalized_char")


def aggregate_vi_to_macro(vi_df, vi_col="vi_normalized"):
    out = []
    for _, r in vi_df.iterrows():
        _, macro, _ = parse_feature_name(r["feature"])
        if macro is not None:
            out.append({"macro": macro, "vi": r[vi_col]})
    if not out:
        return pd.DataFrame(columns=["vi_normalized_macro"])
    s = pd.DataFrame(out).groupby("macro")["vi"].sum()
    tot = s.sum()
    if tot > 0:
        s = s / tot
    return s.sort_values(ascending=False).to_frame("vi_normalized_macro")


# =====================================================================
# 9. Diebold-Mariano with Newey-West SE
# =====================================================================
def newey_west_se(x, lag=NW_LAG):
    x = np.asarray(x, dtype=float)
    n = len(x)
    xc = x - x.mean()
    S = float(np.sum(xc**2)) / n
    for L in range(1, lag + 1):
        w = 1.0 - L / (lag + 1.0)
        cov = float(np.sum(xc[L:] * xc[:-L])) / n
        S += 2.0 * w * cov
    return float(np.sqrt(max(S, 0.0) / n))


def dm_test(pred_a, pred_b, y, dates, lag=NW_LAG, name_a="PCR", name_b="PLS"):
    """Panel DM test from GKX Section 1.8.
      d_t = (1/n_t) Σ_i (e_a,it² − e_b,it²)
      DM   = mean(d) / NW_SE(mean(d))
    Positive DM ⇒ model B has smaller squared error."""
    e2_a = (y - pred_a) ** 2
    e2_b = (y - pred_b) ** 2
    d = pd.DataFrame({"date": dates, "d": e2_a - e2_b}).groupby("date")["d"].mean().sort_index()
    d_vals = d.values
    mean_d = float(d_vals.mean())
    se_nw = newey_west_se(d_vals, lag=lag)
    stat = mean_d / se_nw if se_nw > EPS else np.nan
    from scipy.stats import norm
    p = 2 * (1 - norm.cdf(abs(stat))) if np.isfinite(stat) else np.nan
    return dict(model_a=name_a, model_b=name_b, n_months=len(d_vals),
                mean_d=mean_d, se_nw=se_nw, dm_stat=stat, p_value=p,
                nw_lag=lag,
                reject_5pct_individual=bool(abs(stat) > 1.96) if np.isfinite(stat) else False,
                reject_5pct_bonferroni_12way=bool(abs(stat) > 2.935) if np.isfinite(stat) else False,
                interpretation=(f"{name_b} has smaller error than {name_a}"
                                if mean_d > 0 else
                                f"{name_a} has smaller error than {name_b}"),
                series=d)


# =====================================================================
# 10. Portfolio forecasts (S&P500 proxy + char-sorted)
# =====================================================================
def portfolio_sp500_proxy(preds_df, topN=500):
    """S&P 500 proxy: top-N stocks by ME each month, value-weighted.
    Note: exact S&P 500 membership requires a separate constituent file; this
    proxy is monotone in market cap."""
    df = preds_df.copy()
    df["rnk"] = df.groupby("date")["me"].rank(method="first", ascending=False)
    df = df[df["rnk"] <= topN].copy()
    df["w"] = df["me"] / df.groupby("date")["me"].transform("sum")
    df["wp"] = df["w"] * df["pred"]
    df["wa"] = df["w"] * df["ret_excess"]
    m = df.groupby("date").agg(pred=("wp", "sum"), actual=("wa", "sum"))
    return m, r2_oos(m["actual"].values, m["pred"].values) * 100


def portfolio_char_sorted(preds_df, panel, feature_cols_arr, char_name,
                           n_size=2, n_char=3):
    """Size × char double-sorted portfolios (value-weighted). Char must appear
    as a standalone column in the panel (the pure-characteristic slot)."""
    feat_idx = {c: i for i, c in enumerate(feature_cols_arr)}
    if char_name not in feat_idx:
        return {}, {}

    X_all = panel["X_all"]
    year_arr = panel["panel_year"]; month_arr = panel["panel_month"]
    permno_arr = panel["permno_all"]
    j = feat_idx[char_name]

    # Build lookup of the char value for every test-period row
    yts = panel["year_to_slice"]
    te_rows = [np.arange(*yts[y]) for y in range(FIRST_TEST_YEAR, LAST_TEST_YEAR + 1) if y in yts]
    te_rows = np.concatenate(te_rows)
    lookup = pd.DataFrame({
        "year": year_arr[te_rows], "month": month_arr[te_rows],
        "permno": permno_arr[te_rows], "char_val": X_all[te_rows, j],
    })
    merged = preds_df.merge(lookup, on=["year", "month", "permno"], how="left")
    merged["ym"] = merged["year"].astype(int).astype(str) + "-" + \
                   merged["month"].astype(int).astype(str).str.zfill(2)

    def qbin(s, n):
        try:
            return pd.qcut(s, n, labels=False, duplicates="drop")
        except Exception:
            return pd.Series(np.nan, index=s.index)

    merged["s_bin"] = merged.groupby("ym")["me"].transform(lambda x: qbin(x, n_size))
    merged["c_bin"] = merged.groupby("ym")["char_val"].transform(lambda x: qbin(x, n_char))
    merged = merged.dropna(subset=["s_bin", "c_bin"])
    merged["s_bin"] = merged["s_bin"].astype(int)
    merged["c_bin"] = merged["c_bin"].astype(int)

    labels_s = ["S", "B"] if n_size == 2 else [f"S{i+1}" for i in range(n_size)]
    labels_c = ["L", "M", "H"] if n_char == 3 else [f"C{i+1}" for i in range(n_char)]

    monthly_dict, r2_dict = {}, {}
    for si, sl in enumerate(labels_s):
        for ci, cl in enumerate(labels_c):
            sub = merged[(merged["s_bin"] == si) & (merged["c_bin"] == ci)]
            if len(sub) == 0:
                continue
            w = sub["me"] / sub.groupby("ym")["me"].transform("sum")
            tmp = sub.assign(wp=w * sub["pred"], wa=w * sub["ret_excess"])
            m = tmp.groupby("ym").agg(pred=("wp", "sum"), actual=("wa", "sum"))
            key = f"{sl}{cl}"
            monthly_dict[key] = m
            r2_dict[key] = r2_oos(m["actual"].values, m["pred"].values) * 100
    return monthly_dict, r2_dict


def all_portfolio_forecasts(preds_df, panel, feature_cols_arr, model_name):
    rows = []
    _, r2_sp = portfolio_sp500_proxy(preds_df)
    rows.append(dict(model=model_name, portfolio="SP500_proxy_top500VW", r2_oos_pct=r2_sp))
    for ch in ["bm", "mom12m", "operprof", "agr"]:
        _, r2_map = portfolio_char_sorted(preds_df, panel, feature_cols_arr, ch)
        for pname, r2 in r2_map.items():
            rows.append(dict(model=model_name, portfolio=f"{ch}_{pname}", r2_oos_pct=r2))
    return pd.DataFrame(rows)


# =====================================================================
# 11. Risk-adjusted H-L (FF5+Mom regression + drawdown + turnover)
# =====================================================================
def load_ff_factors():
    if not FF_FILE.exists():
        print(f"  [FF] {FF_FILE} not found — risk-adjusted regression will be skipped.")
        return None
    ff = pd.read_csv(FF_FILE)
    if ff["date"].dtype == object:
        ff["date"] = pd.to_datetime(ff["date"]).dt.to_period("M")
    if ff[["MktRF", "SMB"]].abs().max().max() > 1.0:
        print("  [FF] values appear to be in percent; dividing by 100.")
        for c in ["MktRF", "SMB", "HML", "RMW", "CMA", "Mom", "RF"]:
            if c in ff.columns:
                ff[c] = ff[c] / 100.0
    return ff


def hml_risk_adjusted(dec_monthly, model_name, ff=None):
    if "HML" not in dec_monthly.columns:
        return pd.DataFrame()
    hml = dec_monthly["HML"].dropna()
    cum = np.log1p(hml).cumsum()
    max_dd = (cum.cummax() - cum).max()

    result = dict(model=model_name, weighting="VW",
                  mean_pct=hml.mean() * 100, sd_pct=hml.std() * 100,
                  sr_annualized=(hml.mean() / hml.std()) * np.sqrt(12),
                  max_dd_log=max_dd, worst_month_pct=hml.min() * 100)

    if ff is not None:
        import statsmodels.api as sm
        dates = pd.PeriodIndex(hml.index.astype(str), freq="M")
        ff_al = ff.set_index("date").reindex(dates)
        reg_df = pd.DataFrame({
            "HML": hml.values,
            "MktRF": ff_al["MktRF"].values, "SMB": ff_al["SMB"].values,
            "HMLf":  ff_al["HML"].values,   "RMW": ff_al["RMW"].values,
            "CMA":   ff_al["CMA"].values,   "Mom": ff_al["Mom"].values,
        }).dropna()
        X = sm.add_constant(reg_df[["MktRF", "SMB", "HMLf", "RMW", "CMA", "Mom"]])
        reg = sm.OLS(reg_df["HML"], X).fit(cov_type="HAC", cov_kwds={"maxlags": NW_LAG})
        resid_sd = reg.resid.std()
        result.update(dict(
            alpha_pct_monthly=reg.params["const"] * 100,
            t_alpha=reg.tvalues["const"],
            r2_ff5mom=reg.rsquared,
            info_ratio=(reg.params["const"] / resid_sd) * np.sqrt(12) if resid_sd > 0 else np.nan,
            n_months=int(reg.nobs),
        ))
    return pd.DataFrame([result])


def hml_turnover(preds_df, model_name, top=N_DECILES, bot=1):
    df = preds_df.copy()
    df["_sd"] = df.groupby("date")["pred"].transform("std")
    df = df[df["_sd"] > 0].drop(columns="_sd")
    df["decile"] = df.groupby("date")["pred"].transform(
        lambda x: pd.qcut(x, N_DECILES, labels=False, duplicates="drop"))
    df = df.dropna(subset=["decile"])
    df["decile"] = df["decile"].astype(int) + 1
    df["w_me"] = 0.0
    for d in [top, bot]:
        mask = df["decile"] == d
        df.loc[mask, "w_me"] = (df.loc[mask, "me"] /
                                df.loc[mask].groupby("date")["me"].transform("sum"))
    df["signed_w"] = np.where(df["decile"] == top, df["w_me"],
                              np.where(df["decile"] == bot, -df["w_me"], 0.0))
    pivot = df.pivot_table(index="date", columns="permno",
                           values="signed_w", fill_value=0.0)
    diff = pivot.diff().abs().sum(axis=1)
    return dict(model=model_name, turnover_monthly=float(diff.iloc[1:].mean()),
                n_months=len(diff) - 1)


# =====================================================================
# 12. Campbell-Thompson timing (paper-spec: clip prediction floor at 0
# for long-only; scale by trailing variance; cap leverage at max_leverage)
# =====================================================================
def campbell_thompson_timing(portfolio_monthly, model_name,
                              max_leverage=0.5, no_short=True):
    if "pred" not in portfolio_monthly or "actual" not in portfolio_monthly:
        return {}
    d = portfolio_monthly.dropna().copy()
    if len(d) < 60:
        return {}

    # Campbell & Thompson (2008) spec:
    #   (a) clip predicted excess return floor at 0 for long-only
    #   (b) expanding (past-only) variance σ̂²_t of the target return
    #   (c) weight w_t = pred_t / (γ · σ̂²_t), fixed γ implicit via cap
    #   (d) cap |w_t| at 1 + max_leverage; for long-only clip at [0, 1+max_leverage]
    pred_for_timing = d["pred"].clip(lower=0) if no_short else d["pred"]
    d["var_past"] = d["actual"].expanding(min_periods=24).var().shift(1)
    d = d.dropna(subset=["var_past"])
    if d.empty:
        return {}

    raw = pred_for_timing.loc[d.index] / d["var_past"]
    # Normalize so that buy-and-hold (w=1 always) corresponds to raw scale ≈ 1
    scale = raw.abs().mean()
    if scale > 0:
        raw = raw / scale
    low = 0.0 if no_short else -max_leverage
    high = 1.0 + max_leverage
    w = raw.clip(low, high)
    timed_ret = w * d["actual"]

    def sr(r):
        return r.mean() / r.std() * np.sqrt(12) if r.std() > 0 else np.nan

    return dict(model=model_name,
                sr_buyhold=sr(d["actual"]), sr_timed=sr(timed_ret),
                sr_gain=sr(timed_ret) - sr(d["actual"]),
                n_months=len(d))


# =====================================================================
# 13. Marginal-effect curves
# =====================================================================
def marginal_effect_curves(last_window, feature_cols_arr, model_name,
                            char_names=MARGINAL_CHARS, n_grid=21):
    """Sweep one characteristic from -1 to +1; hold all others at training-mean
    (≈ 0 after rank-standardization in the panel). Model-implied expected return."""
    if last_window is None or last_window["best_k"] == 0:
        return pd.DataFrame()

    loadings = last_window["loadings"][:, :last_window["best_k"]]
    beta = last_window["beta"]
    mu, sigma = last_window["mu"], last_window["sigma"]
    P = len(feature_cols_arr)
    idx = {c: i for i, c in enumerate(feature_cols_arr)}
    grid = np.linspace(-1.0, 1.0, n_grid)

    base_z = (-mu / sigma).astype(np.float64)    # z at raw x = 0

    rows = []
    for ch in char_names:
        if ch not in idx:
            continue
        j = idx[ch]
        for v in grid:
            z = base_z.copy()
            z[j] = (v - mu[j]) / sigma[j]
            score = z @ loadings.astype(np.float64)
            rows.append(dict(model=model_name, char=ch, x_value=float(v),
                             expected_return=float(score @ beta)))
    return pd.DataFrame(rows)


# =====================================================================
# 14. Google Drive stock-level prediction export
# =====================================================================
def mount_google_drive_for_results():
    drive_root = Path("/content/drive/MyDrive")
    try:
        from google.colab import drive
        if not drive_root.exists():
            drive.mount("/content/drive")
    except ImportError:
        if not drive_root.exists():
            print("Google Drive is not available; skipping Drive prediction export.")
            return None
    return drive_root if drive_root.exists() else None


def format_stock_level_predictions_for_export(model_name, preds_df, yearly_df):
    prediction_source = "pcr_pls"
    source_notebook = "Notebooks for Methodology/PCR & PLS.ipynb"
    model_label = {"pcr": "PCR", "pls": "PLS"}.get(model_name, model_name.upper())

    export_base = preds_df.copy()
    export_base["date"] = pd.to_datetime(export_base["date"])
    export_base["test_year"] = export_base["date"].dt.year.astype("int16")
    if "exchcd" not in export_base.columns:
        export_base["exchcd"] = np.nan

    yearly_cols = [
        c for c in ["test_year", "best_k", "val_r2_pct", "ceiling_hit", "n_train", "n_valid", "n_test"]
        if c in yearly_df.columns
    ]
    if yearly_cols:
        export_base = export_base.merge(yearly_df[yearly_cols], on="test_year", how="left")
    for col in ["best_k", "val_r2_pct", "ceiling_hit", "n_train", "n_valid", "n_test"]:
        if col not in export_base.columns:
            export_base[col] = np.nan

    return pd.DataFrame({
        "source_notebook": source_notebook,
        "source_panel": str(PANEL_FILE),
        "prediction_source": prediction_source,
        "prediction_scope": "stock_level_oos_with_ids_for_bottom_up_portfolios",
        "model": model_label,
        "test_year": export_base["test_year"],
        "date": export_base["date"],
        "permno": export_base["permno"],
        "exchcd": export_base["exchcd"],
        "me": export_base["me"],
        "ret_excess": export_base["ret_excess"],
        "actual": export_base["ret_excess"],
        "prediction": export_base["pred"],
        "predicted": export_base["pred"],
        "best_k": export_base["best_k"],
        "validation_r2_pct": export_base["val_r2_pct"],
        "ceiling_hit": export_base["ceiling_hit"],
        "n_train": export_base["n_train"],
        "n_valid": export_base["n_valid"],
        "n_test": export_base["n_test"],
        "active_features": np.nan,
    })


def save_drive_stock_level_predictions(all_preds, all_yearly):
    drive_root = mount_google_drive_for_results()
    if drive_root is None:
        return None

    prediction_source = "pcr_pls"
    source_notebook = "Notebooks for Methodology/PCR & PLS.ipynb"
    source_panel = str(PANEL_FILE)
    output_dir = drive_root / "Replication Paper/Data/Results" / prediction_source
    output_dir.mkdir(parents=True, exist_ok=True)

    frames = [
        format_stock_level_predictions_for_export(model_name, all_preds[model_name], all_yearly[model_name])
        for model_name in ["pcr", "pls"]
        if model_name in all_preds and model_name in all_yearly
    ]
    if not frames:
        print("No PCR/PLS predictions available for Drive export.")
        return None

    oos_predictions = pd.concat(frames, ignore_index=True)
    oos_predictions = oos_predictions.sort_values(["date", "permno", "model"]).reset_index(drop=True)

    prediction_path = output_dir / f"{prediction_source}_stock_level_oos_predictions_with_ids.parquet"
    manifest_path = output_dir / f"{prediction_source}_prediction_manifest.csv"

    oos_predictions.to_parquet(prediction_path, index=False)

    manifest = pd.DataFrame([{
        "prediction_source": prediction_source,
        "source_notebook": source_notebook,
        "source_panel": source_panel,
        "model": ",".join(sorted(oos_predictions["model"].unique())),
        "file_type": "stock_level_oos_predictions_with_ids",
        "path": str(prediction_path),
        "rows": len(oos_predictions),
        "columns": ",".join(oos_predictions.columns),
    }])
    manifest.to_csv(manifest_path, index=False)

    print(f"Saved PCR/PLS stock-level OOS predictions: {prediction_path}")
    print(f"Saved PCR/PLS prediction manifest: {manifest_path}")
    print(f"Prediction panel shape: {oos_predictions.shape}")
    print(oos_predictions[["model", "permno", "date", "ret_excess", "prediction", "me", "exchcd"]].head())
    return oos_predictions, manifest


# =====================================================================
# 15. Main pipeline
# =====================================================================
def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--skip-annual", action="store_true",
                   help="Skip the 12-month-ahead backtest (saves ~30 min).")
    p.add_argument("--skip-vi", action="store_true",
                   help="Skip variable importance (saves 5-15 min at P=920).")
    p.add_argument("--skip-portfolios", action="store_true")
    p.add_argument("--k-ensemble", type=int, default=K_ENSEMBLE_DELTA,
                   help="K-ensemble delta. 0 = single-K, 3 = recommended smoother.")
    return p.parse_known_args(argv)[0]


def main():
    args = parse_args()
    t_total = time.time()

    panel = load_panel()
    print(f"\nPanel: N={panel['N']:,}, P={panel['P']}, "
          f"years {min(panel['year_to_slice'])}-{max(panel['year_to_slice'])}")

    all_table1, all_preds, all_yearly = [], {}, {}
    all_last_window, all_nonzero = {}, {}
    all_dec_stats, all_dec_monthly = [], {}
    all_r2_curves = {}

    for model_name, max_k in [("pcr", MAX_K_PCR), ("pls", MAX_K_PLS)]:
        preds_df, yearly_df, r2_curves_df, last_win, nonzero_wins = run_backtest(
            model_name, max_k, panel, k_ensemble_delta=args.k_ensemble)

        all_preds[model_name] = preds_df
        all_yearly[model_name] = yearly_df
        all_last_window[model_name] = last_win
        all_nonzero[model_name] = nonzero_wins
        all_r2_curves[model_name] = r2_curves_df

        t1 = evaluate_table1(preds_df, model_name)
        all_table1.append(t1)
        print(f"\n  {model_name.upper()} Table 1:")
        for _, r in t1.iterrows():
            print(f"    {r['subsample']:>15s}: {r['r2_oos_pct']:+.4f}% (n={r['n_obs']:,})")

        # Save core artifacts
        preds_df.to_csv(OUTPUT_DIR / f"{model_name}_predictions.csv", index=False)
        yearly_df.to_csv(OUTPUT_DIR / f"{model_name}_yearly_summary.csv", index=False)
        r2_curves_df.to_csv(OUTPUT_DIR / f"k_stability_{model_name}.csv", index=False)

        evaluate_annual_calendar(preds_df, model_name).to_csv(
            OUTPUT_DIR / f"{model_name}_annual_r2_calendar.csv", index=False)

        dec_stats, dec_monthly = decile_portfolio(preds_df, model_name)
        all_dec_stats.append(dec_stats)
        all_dec_monthly[model_name] = dec_monthly
        dec_stats.to_csv(OUTPUT_DIR / f"{model_name}_decile_stats.csv", index=False)
        dec_monthly.to_csv(OUTPUT_DIR / f"{model_name}_decile_monthly.csv")

    # Save one combined PCR/PLS prediction panel to Google Drive, matching the GLM export style.
    save_drive_stock_level_predictions(all_preds, all_yearly)

    # Table 1
    pd.concat(all_table1, ignore_index=True).to_csv(
        OUTPUT_DIR / "table1_r2_summary.csv", index=False)

    # -------------------------------------------------------------
    # Table 2 proper: 12-month-ahead pipeline
    # -------------------------------------------------------------
    if not args.skip_annual:
        print(f"\n{'='*70}\nBUILDING 12-MONTH-AHEAD TARGET\n{'='*70}")
        y_ann = build_annual_y(panel)
        n_ok = int(np.isfinite(y_ann).sum())
        print(f"  non-NaN rows: {n_ok:,} ({100*n_ok/len(y_ann):.1f}%)")

        ann_frames = []
        for model_name, max_k in [("pcr", MAX_K_PCR), ("pls", MAX_K_PLS)]:
            ann_r2, ann_yr = run_backtest_annual(model_name, max_k, panel, y_ann)
            ann_frames.append(ann_r2)
            ann_yr.to_csv(OUTPUT_DIR / f"{model_name}_annual_yearly_summary.csv", index=False)
        pd.concat(ann_frames, ignore_index=True).to_csv(
            OUTPUT_DIR / "table2_annual_r2_12m_ahead.csv", index=False)

    # -------------------------------------------------------------
    # Variable importance
    # -------------------------------------------------------------
    if not args.skip_vi:
        print(f"\n{'='*70}\nVARIABLE IMPORTANCE\n{'='*70}")

        print("\n[PCR] feature-level VI (GKX spec: freeze loadings, zero column j)")
        vi_pcr = variable_importance_pcr(all_last_window["pcr"],
                                          panel["X_all"], panel["y_all"],
                                          panel["feature_cols_arr"],
                                          recompute_pcs=PCR_VI_RECOMPUTE_PCS)
        vi_pcr.to_csv(OUTPUT_DIR / "pcr_variable_importance.csv", index=False)
        if not vi_pcr.empty:
            aggregate_vi_to_char(vi_pcr).to_csv(OUTPUT_DIR / "pcr_char_importance.csv")
            aggregate_vi_to_macro(vi_pcr).to_csv(OUTPUT_DIR / "pcr_macro_importance.csv")
            print("  PCR top 20 characteristics:")
            print(aggregate_vi_to_char(vi_pcr).head(20).to_string())

        print("\n[PLS] feature-level VI (average |R·beta| across ALL refits with K>0)")
        vi_pls = variable_importance_pls(all_nonzero["pls"], panel["feature_cols_arr"])
        vi_pls.to_csv(OUTPUT_DIR / "pls_variable_importance.csv", index=False)
        if not vi_pls.empty:
            aggregate_vi_to_char(vi_pls).to_csv(OUTPUT_DIR / "pls_char_importance.csv")
            aggregate_vi_to_macro(vi_pls).to_csv(OUTPUT_DIR / "pls_macro_importance.csv")
            print("  PLS top 20 characteristics:")
            print(aggregate_vi_to_char(vi_pls).head(20).to_string())

    # -------------------------------------------------------------
    # DM test
    # -------------------------------------------------------------
    print(f"\n{'='*70}\nDIEBOLD-MARIANO TEST (Newey-West lag={NW_LAG})\n{'='*70}")
    merged = all_preds["pcr"].merge(
        all_preds["pls"][["permno", "date", "pred"]],
        on=["permno", "date"], suffixes=("_pcr", "_pls"))
    dm = dm_test(merged["pred_pcr"].values, merged["pred_pls"].values,
                 merged["ret_excess"].values, merged["date"].values,
                 lag=NW_LAG, name_a="PCR", name_b="PLS")
    dm["series"].to_frame("d_pcr_vs_pls").to_csv(
        OUTPUT_DIR / "dm_series_pcr_vs_pls.csv")
    dm_summary = {k: v for k, v in dm.items() if k != "series"}
    with open(OUTPUT_DIR / "dm_test_pcr_vs_pls.json", "w") as f:
        json.dump(dm_summary, f, indent=2, default=str)
    print(f"  DM stat = {dm['dm_stat']:+.3f}  p = {dm['p_value']:.4f}")
    print(f"  {dm['interpretation']}")
    print(f"  Reject 5% individual : {dm['reject_5pct_individual']}")
    print(f"  Reject 5% Bonferroni : {dm['reject_5pct_bonferroni_12way']}")

    # -------------------------------------------------------------
    # Portfolio forecasts + risk-adjusted + timing
    # -------------------------------------------------------------
    if not args.skip_portfolios:
        print(f"\n{'='*70}\nPORTFOLIO FORECASTS\n{'='*70}")
        port_frames = []
        for model_name in ["pcr", "pls"]:
            port_frames.append(all_portfolio_forecasts(
                all_preds[model_name], panel, panel["feature_cols_arr"], model_name))
        pd.concat(port_frames, ignore_index=True).to_csv(
            OUTPUT_DIR / "portfolio_forecasts.csv", index=False)

        print(f"\n{'='*70}\nRISK-ADJUSTED H-L PERFORMANCE\n{'='*70}")
        ff = load_ff_factors()
        risk_frames, turn_rows = [], []
        for model_name in ["pcr", "pls"]:
            risk_frames.append(hml_risk_adjusted(all_dec_monthly[model_name],
                                                  model_name, ff=ff))
            turn_rows.append(hml_turnover(all_preds[model_name], model_name))
        risk_full = (pd.concat(risk_frames, ignore_index=True)
                       .merge(pd.DataFrame(turn_rows), on="model", how="left"))
        risk_full.to_csv(OUTPUT_DIR / "risk_adjusted_performance.csv", index=False)
        print(risk_full.to_string())

        print(f"\n{'='*70}\nCAMPBELL-THOMPSON TIMING (S&P500 proxy)\n{'='*70}")
        timing_rows = []
        for model_name in ["pcr", "pls"]:
            sp_m, _ = portfolio_sp500_proxy(all_preds[model_name])
            t_ = campbell_thompson_timing(sp_m, model_name)
            if t_:
                timing_rows.append(t_)
        if timing_rows:
            pd.DataFrame(timing_rows).to_csv(
                OUTPUT_DIR / "timing_strategy_sharpe_gain.csv", index=False)
            print(pd.DataFrame(timing_rows).to_string(index=False))

    # -------------------------------------------------------------
    # Marginals
    # -------------------------------------------------------------
    print(f"\n{'='*70}\nMARGINAL EFFECT CURVES\n{'='*70}")
    for model_name in ["pcr", "pls"]:
        lw = all_last_window[model_name]
        if lw is None or lw["best_k"] == 0:
            lw = all_nonzero[model_name][-1] if all_nonzero[model_name] else None
        if lw is None:
            continue
        m = marginal_effect_curves(lw, panel["feature_cols_arr"], model_name)
        if not m.empty:
            m.to_csv(OUTPUT_DIR / f"marginals_{model_name}.csv", index=False)
            print(f"  [{model_name.upper()}] marginals for: {sorted(m['char'].unique())}")

    print(f"\n{'='*70}\nDONE. Total {(time.time()-t_total)/60:.1f} min.\n"
          f"Outputs in: {OUTPUT_DIR}\n{'='*70}")


if __name__ == "__main__":
    main()


LOADING PANEL
  rows       : 3,305,648
  pre-sorted : True
  features   : P=278
  X_all size : 3.42 GB (float32)
    278/278 columns loaded
  years      : 1957-2021
  stocks     : 25,770

Panel: N=3,305,648, P=278, years 1957-2021

Running PCR (max K = 60, K-ensemble delta = 0)
  [PCR] 1987 | K*= 29 | val_R²=+0.2375% | ceiling=False |   2.6s
  [PCR] 1988 | K*= 28 | val_R²=+0.1913% | ceiling=False |   2.7s
  [PCR] 1989 | K*= 31 | val_R²=+0.2297% | ceiling=False |   2.8s
  [PCR] 1990 | K*= 31 | val_R²=+0.2088% | ceiling=False |   3.1s
  [PCR] 1991 | K*= 28 | val_R²=+0.1685% | ceiling=False |   3.2s
  [PCR] 1992 | K*= 28 | val_R²=+0.1429% | ceiling=False |   3.3s
  [PCR] 1993 | K*= 29 | val_R²=+0.1002% | ceiling=False |   3.6s
  [PCR] 1994 | K*= 16 | val_R²=+0.0838% | ceiling=False |   3.6s
  [PCR] 1995 | K*= 47 | val_R²=+0.2105% | ceiling=False |   4.5s
  [PCR] 1996 | K*= 55 | val_R²=+0.2166% | ceiling=False |   5.2s
  [PCR] 1997 | K*= 55 | val_R²=+0.2215% | ceiling=False |   6.2s
  [PC